**Research Question: How does sleep quality affect stress levels (HRV measurements) in COVID-19 patients?**

- Understand your data - what do the features mean? (May have to do some info gathering)
  - Datasets I chose: hrv_measurements.csv, and sleep.csv
  - Key features to look for:
    - hrv_measurements.csv
      - user_code: unique patient ID
      - measurement_datetime: timestamp of recording
      - time_of_day: morning/evening, etc.
      - bpm: beats per minute during recording
      - meanrr: mean RR interval
      - sdnn: standard deviation of NN intervals → HRV measure (stress resilience)
      - rmssd: root mean square of successive differences → HRV measure (parasympathetic activity)
      - pnn50: % of RR intervals >50ms
      - lf, hf, vlf: spectral HRV components (low, high, very low frequency)
      - lfhf: LF/HF ratio
      - total_power: overall HRV signal strength
      - how_feel, how_mood, how_sleep: self-reported well-being
    - sleep.csv
      - user_code: unique patient ID
      - day: date of sleep
      - sleep_duration: total time asleep
      - sleep_rem_duration, sleep_light_duration, sleep_deep_duration: time spent in sleep stages
      - sleep_awake_duration: awake time during sleep
      - pulse_min, pulse_max, pulse_average: heart rate measurements during sleep
- Document data context and data sampling in markdown
  - Context: Collected from Welltory app; all participants tested positive for COVID-19
  - HRV data: Recorded at specific times of day
  - Sleep data: Recorded nightly by wearables
  - Problems with data: Not all users recorded both HRV and sleep every day → dataset is incomplete and uneven
- Explore and interpret data structure, descriptive statistics, data quality, and variable relationships
- Explore data visually with appropriate visualizations
- Discuss and implement strategies for Handling Missing Values, Removing Duplicates, and Handling Outliers
  - Drop duplicates by: user_code + day
  - Missing values: if sleep missing, that night is excluded
  - Outliers: set a realistic range for sleep_duration (2-20 hours), drop rmssd (hrv) that is 0 or below
- Perform data transformation as appropriate
- Create at least one new feature and document your approach
  - New feature would have been a 3-day rolling mean of rmssd to smooth fluctuations as it captures physiological stress recovery patterns
- Include a discussion around data quality assessment, including data profiling, data completeness, data accuracy, data consistency, data integrity, and data lineage and provenance
  - Many missing nights/HRV logs (uneven coverage)
  - Sleep stage detection may vary by device; HRV depends on sensor quality
  - Need to ensure user_code aligns across datasets
  - Some users may have HRV records without sleep data therefore, datasets must be merged carefully

In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
hrv_file = "/Users/arnavmahale/Documents/Duke University/Fall '25/AIPI 510/aipi510-fall25/data/week4/hrv_measurements.csv"
hrv_data = pd.read_csv(hrv_file)
# print(hrv_data.info())

sleep_file = "/Users/arnavmahale/Documents/Duke University/Fall '25/AIPI 510/aipi510-fall25/data/week4/sleep.csv"
sleep_data = pd.read_csv(sleep_file)
# print(sleep_data.info())


# Cleaning and Preprocessing
# convert datetime fields
hrv_data["measurement_datetime"] = pd.to_datetime(hrv_data["measurement_datetime"], errors="coerce")
sleep_data["day"] = pd.to_datetime(sleep_data["day"], errors="coerce")

# create a 'day' column in HRV for alignment
hrv_data["day"] = hrv_data["measurement_datetime"].dt.date
hrv_data["day"] = pd.to_datetime(hrv_data["day"])

# convert sleep duration to hours
med = sleep_data["sleep_duration"].median()

# unit detection
if med > 1000:
    unit = "seconds"
    sleep_data["sleep_hours"] = sleep_data["sleep_duration"] / 3600.0
elif med > 20:
    unit = "minutes"
    sleep_data["sleep_hours"] = sleep_data["sleep_duration"] / 60.0
else:
    unit = "hours"
    sleep_data["sleep_hours"] = sleep_data["sleep_duration"]

print(sleep_data["sleep_hours"].describe(percentiles=[.05,.25,.5,.75,.95]))

# remove outliers
sleep_data = sleep_data[(sleep_data["sleep_hours"] >= 2) & (sleep_data["sleep_hours"] <= 20)]
hrv_data = hrv_data[hrv_data["rmssd"] > 0]

print("HRV shape after cleaning:", hrv_data.shape)
print("Sleep shape after cleaning:", sleep_data.shape)

count    425.000000
mean       7.138814
std        2.159390
min        0.258333
5%         3.340889
25%        6.071389
50%        7.233611
75%        8.500000
95%       10.310000
max       13.183333
Name: sleep_hours, dtype: float64
HRV shape after cleaning: (3245, 23)
Sleep shape after cleaning: (410, 13)


Wasn't able to complete due to time restrictions. But next steps essentially were:
- Once the data has been cleaned, merge the two datasets (keeping all relevant features)
- Define sleep quality categories
- Explore and visualize
- Statistical testing